# RAG Workshop: from PDF to answer

**Run it:** click each cell and press ▶ (or Shift+Enter), top to bottom. No setup needed.

```
PDFs → load → chunk → embed → vector store ──┐
                                              ├→ retrieve top-k → prompt (context + question) → LLM → answer
question → embed ─────────────────────────────┘
```

## 0. Install (about 1 to 2 minutes the first time)

> Red lines starting with `ERROR: pip's dependency resolver...` are expected in Colab. They only mention Colab's own tools, so ignore them and continue. If a later cell shows an import error, choose **Runtime → Restart session** and run again from step 1 (skip this install cell).

In [ ]:
!pip install -q langchain-community langchain-text-splitters pypdf chromadb sentence-transformers langchain-openai langchain-google-genai pandas

## 1. Data ingestion: read documents into Python

A `Document` is just text plus **metadata** (where it came from, which page).
Upload your own PDFs or .txt files. If you skip the upload, a small Boston example is used.

In [ ]:
import os, glob
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("my_files"); DATA_DIR.mkdir(exist_ok=True)

try:                                   # inside Colab: open the upload dialog
    from google.colab import files
    for name, content in files.upload().items():
        (DATA_DIR / name).write_bytes(content)
except ImportError:                    # running locally
    pass

docs = []
for path in sorted(DATA_DIR.glob("**/*")):
    if path.suffix.lower() == ".pdf":
        pages = PyPDFLoader(str(path)).load()
    elif path.suffix.lower() in {".txt", ".md"}:
        pages = TextLoader(str(path), encoding="utf-8").load()
    else:
        continue
    for p in pages:
        p.metadata["source_file"] = path.name
    docs.extend(pages)

if not docs:   # nothing uploaded: use a built-in example
    sample = [
        "The Freedom Trail is a 2.5 mile red brick line through Boston connecting 16 historic sites from the American Revolution.",
        "The USS Constitution is a famous ship nicknamed Old Ironsides because it has never been defeated in battle.",
        "The Museum of Science has interactive exhibits, a butterfly garden, 4-D films, IMAX films and planetarium shows.",
        "The Isabella Stewart Gardner Museum was the site of the biggest unsolved art theft in history, in 1990.",
        "Faneuil Hall has stood since 1742 and is home to shops and restaurants. Quincy Market is right behind it.",
        "The New England Aquarium is a research and conservation group with a Giant Ocean Tank.",
    ]
    docs = [Document(page_content=t, metadata={"source_file": "boston_example", "page": i}) for i, t in enumerate(sample)]

print(f"Loaded {len(docs)} pages/documents")
docs[0]

## 2. Chunking: cut the text into pieces

Long texts do not fit in one search, so we cut them into **chunks**.
`chunk_overlap` repeats a few characters at the start of the next chunk so sentences are not cut in half.

Try changing `CHUNK_SIZE` and `CHUNK_OVERLAP` and re-run this cell and the ones below.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],   # try paragraphs first, then lines, then words
)
chunks = splitter.split_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks")

pd.set_option("display.max_colwidth", 110)
pd.DataFrame({
    "file": [c.metadata["source_file"] for c in chunks],
    "page": [c.metadata.get("page") for c in chunks],
    "chars": [len(c.page_content) for c in chunks],
    "text": [c.page_content for c in chunks],
}).head(10)

## 3. Embeddings: turn text into numbers

An **embedding** is a list of numbers (here 384) that captures meaning.
Texts with similar meaning get similar numbers, even with different words.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c.page_content for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
print("shape:", embeddings.shape, "(chunks x numbers per chunk)")
print("first 8 numbers of chunk 0:", embeddings[0][:8].round(3))

In [ ]:
# Meaning, not matching words: "boat" vs "ship" share no letters, but are close in meaning.
pairs = [("a big boat on the sea", "a large ship in the ocean"),
         ("a big boat on the sea", "my favourite pizza topping")]
for a, b in pairs:
    va, vb = model.encode([a, b], normalize_embeddings=True)
    print(f"similarity = {float(va @ vb):.2f}   {a!r} vs {b!r}")

## 4. Vector store: save the vectors so we can search them

We use **ChromaDB** (in memory here). We ask for **cosine** distance, so `similarity = 1 - distance`.

In [ ]:
import chromadb, uuid

client = chromadb.Client()
try:
    client.delete_collection("workshop")
except Exception:
    pass
collection = client.create_collection("workshop", metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{k: v for k, v in c.metadata.items() if isinstance(v, (str, int, float))} for c in chunks],
)
print("chunks stored:", collection.count())

### 4b. Look inside the vector database

A vector database is a table. Every row holds an **id**, the original **text**, its **metadata** and its **vector** (the embedding).
Let's open it up and look.

In [ ]:
rows = collection.get(include=["documents", "metadatas", "embeddings"], limit=5)

pd.set_option("display.max_colwidth", 70)
pd.DataFrame({
    "id": rows["ids"],
    "source_file": [m.get("source_file") for m in rows["metadatas"]],
    "page": [m.get("page") for m in rows["metadatas"]],
    "text": rows["documents"],
    "vector_length": [len(v) for v in rows["embeddings"]],
    "first_5_numbers": [[round(float(x), 3) for x in v[:5]] for v in rows["embeddings"]],
})

**Metadata filters:** a vector database can also filter like a normal database.
Here we ask only for rows that came from one file.

In [ ]:
first_file = chunks[0].metadata["source_file"]
filtered = collection.get(where={"source_file": first_file}, limit=3, include=["documents", "metadatas"])
print(f"Rows from {first_file!r}: showing {len(filtered['ids'])}")
for i, doc in zip(filtered["ids"], filtered["documents"]):
    print(i, "->", doc[:90].replace("\n", " "))

**Saving to disk:** the store above lives in memory and disappears when Colab restarts.
A `PersistentClient` writes it to a folder, so you can close the notebook, come back and reconnect.

In [ ]:
import sqlite3

DB_PATH = "my_vector_db"
disk = chromadb.PersistentClient(path=DB_PATH)
try:
    disk.delete_collection("workshop")
except Exception:
    pass
saved = disk.create_collection("workshop", metadata={"hnsw:space": "cosine"})
saved.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{k: v for k, v in c.metadata.items() if isinstance(v, (str, int, float))} for c in chunks],
)

print("Files on disk:")
for root, _, names in os.walk(DB_PATH):
    for n in names:
        path = os.path.join(root, n)
        print(f"  {path}  ({os.path.getsize(path):,} bytes)")

# Chroma keeps the text and metadata in a normal SQLite file. Here are its tables:
con = sqlite3.connect(os.path.join(DB_PATH, "chroma.sqlite3"))
print("\nSQLite tables:", [t[0] for t in con.execute("select name from sqlite_master where type='table'")])
con.close()

# Reconnect with a brand-new client, as if you had restarted:
again = chromadb.PersistentClient(path=DB_PATH).get_collection("workshop")
print("\nReconnected. Rows found:", again.count())

## 5. Retrieval: find the most relevant chunks

Embed the **question** the same way, then ask the store for the nearest chunks.

In [ ]:
def retrieve(query, top_k=3, score_threshold=0.0):
    q = model.encode([query], normalize_embeddings=True)[0]
    res = collection.query(query_embeddings=[q.tolist()], n_results=top_k)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        score = 1 - dist
        if score >= score_threshold:
            hits.append({"score": round(score, 3), "source": meta.get("source_file"),
                         "page": meta.get("page"), "content": doc})
    return hits

question = "Which ship is nicknamed Old Ironsides?"   # <- change me!
pd.DataFrame(retrieve(question, top_k=3))

**A map of the vector store:** each dot is one chunk. We squash the 384 numbers into 2 so we can draw them, so only the *distances* matter, not the axis values.
Chunks about similar things sit close together. The **star** is your question. The **red rings** are the chunks the search picked. They are the same chunks as in the table above.
The picture is a rough 2D squash, so a ring is not always the dot visually nearest the star. The scores in the table are the exact answer.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

TOP_K = 3
pca = PCA(n_components=2).fit(embeddings)
xy = pca.transform(embeddings)
files = [c.metadata["source_file"] for c in chunks]

# The chunks the search picked: the ones whose full 384-number vectors are closest to the question
q_vec = model.encode([question], normalize_embeddings=True)
top = (embeddings @ q_vec[0]).argsort()[::-1][:TOP_K]
q_xy = pca.transform(q_vec)[0]

fig, ax = plt.subplots(figsize=(8, 6))
for f in sorted(set(files)):
    idx = [i for i, x in enumerate(files) if x == f]
    ax.scatter(xy[idx, 0], xy[idx, 1], label=f[:30], s=60, alpha=0.6)

for rank, i in enumerate(top, 1):
    ax.plot([q_xy[0], xy[i, 0]], [q_xy[1], xy[i, 1]], "k--", lw=1)                       # line from the question
    ax.scatter(xy[i, 0], xy[i, 1], s=260, facecolors="none", edgecolors="red", linewidths=2.5)   # red ring
    ax.annotate(f"#{rank} (chunk {i})", xy[i], textcoords="offset points", xytext=(10, 8), color="red", weight="bold")

ax.scatter(q_xy[0], q_xy[1], marker="*", s=450, c="black", label="your question", zorder=5)
ax.set_title(f"Red rings = the {TOP_K} chunks retrieved for: {question[:50]}")
ax.margins(0.2)   # room for the labels
ax.legend(fontsize=8)
plt.show()

## 6. Generation: give the context to an LLM

The best chunks are the **context**. The **prompt** = instruction + context + question.
The instruction "say I don't know" stops the model from making things up.

**Get a free Gemini key (no credit card):** click the key icon in the left sidebar of Colab, choose **Secrets**, open **Gemini API keys**, create a key, and turn on **Notebook access** for `GEMINI_API_KEY`. Prefer OpenAI? Set `PROVIDER = "openai"` in the cell below and use an `OPENAI_API_KEY` secret.

In [ ]:
def build_prompt(context, question):
    return f"""Use the following context to answer the question concisely.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

hits = retrieve(question, top_k=3)
context = "\n\n".join(h["content"] for h in hits)
prompt = build_prompt(context, question)
print(prompt)

In [ ]:
# Pick the AI provider. "gemini" is free (get a key from the Colab Secrets panel, see the note above).
PROVIDER = "gemini"        # or "openai"
GEMINI_MODEL = "gemini-3.5-flash-lite"   # a fast, light model. Not working? Run the next cell to list the models your key can use.
OPENAI_MODEL = "gpt-4o-mini"

import os
from getpass import getpass

KEY_NAME = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY"}[PROVIDER]

def get_api_key(name):
    """Environment variable -> Colab Secret -> hidden prompt. Never saved in the notebook."""
    if os.getenv(name):
        return os.environ[name]
    try:
        from google.colab import userdata          # Colab sidebar: key icon -> Secrets
        return userdata.get(name)
    except Exception:
        return getpass(f"{name}: ")

api_key = get_api_key(KEY_NAME)
print("API key found:", bool(api_key))

def list_gemini_models(api_key):
    """Print every Gemini model this key can use for chat (generateContent)."""
    from google import genai
    client = genai.Client(api_key=api_key)   # keep this variable: the client closes when it is garbage-collected
    for m in client.models.list():
        if "generateContent" in (m.supported_actions or []):
            print(m.name.replace("models/", ""))

print("Provider:", PROVIDER, "| Model:", GEMINI_MODEL if PROVIDER == "gemini" else OPENAI_MODEL)

if PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, google_api_key=api_key, temperature=0.1, max_output_tokens=512, timeout=30, max_retries=1)
else:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=OPENAI_MODEL, api_key=api_key, temperature=0.1, max_tokens=512, timeout=30, max_retries=1)

def reply_text(message):
    """Some models return a list of content blocks instead of a plain string."""
    c = message.content
    return c if isinstance(c, str) else "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in c)

print(reply_text(llm.invoke(prompt)))

**Which models can I use?** Model names change over time. If the cell above shows a "404 model not found" error, or you are just curious, run this cell to see the models your key can use. Copy a name into `GEMINI_MODEL` above, then re-run the cell above.

In [ ]:
if PROVIDER == "gemini":
    list_gemini_models(api_key)
else:
    print("This list is for Gemini. For OpenAI see https://platform.openai.com/docs/models")

## 7. Put it together and experiment

Change the question, `top_k`, or the chunk size, and see whether the answer improves.
Then ask something the documents **cannot** answer and check for "I don't know".

In [ ]:
def ask(question, top_k=3):
    hits = retrieve(question, top_k=top_k)
    context = "\n\n".join(h["content"] for h in hits)
    answer = reply_text(llm.invoke(build_prompt(context, question)))
    print("Q:", question, "\nA:", answer, "\n\nSources:")
    for h in hits:
        print(f"  {h['score']:.2f}  {h['source']} (page {h['page']})")

ask("Which museum has a butterfly garden?")
ask("What is the attention mechanism in transformers?")   # not in the documents